# MimicVision AI v0.5 - Clase 2 | MATCH

Segunda entrega: recuperacion visual por similitud. Reutiliza el dataset y el pipeline
de MIMIC (Clase 1) sin curar nada nuevo, y compara tres representaciones distintas para
recuperar imagenes parecidas a una consulta: descriptores clasicos (HOG + LBP),
geometria de pose heredada de MIMIC, y embeddings visuales modernos (SigLIP2).

Corre igual en local y en Colab, igual que el notebook de la Clase 1.

In [1]:
# Mismo patron de deteccion de entorno que D1: si google.colab existe,
# estamos en Colab y hay que clonar e instalar; si no, asumimos que el
# entorno local ya tiene requirements.txt instalado.
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if not Path("mimicvision-ai").exists() and not Path("mimic").exists():
        !git clone https://github.com/USUARIO/mimicvision-ai.git
    if Path("mimicvision-ai").exists():
        os.chdir("mimicvision-ai")
    !pip install -q -r requirements.txt

if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Entorno:", "Colab" if EN_COLAB else "local", "| Carpeta:", Path.cwd().name)

Entorno: local | Carpeta: computer_vision


## 1. Galeria y consultas

Punto de partida obligatorio: el dataset de MIMIC (`data/metadata.csv`) ya tiene 330
imagenes en 10 clases con un split fijo -- de sobra para el minimo de la Clase 2 (300
imagenes, 6 categorias). No se cura galeria nueva.

Restriccion importante para que la comparacion entre rutas sea justa: la ruta B
(geometria) solo existe para las imagenes donde MIMIC detecto landmarks (275 de 330,
seccion 2 de D1). Si cada ruta evaluara sobre un conjunto distinto de imagenes, las
diferencias de Recall no se podrian atribuir solo a la representacion. Por eso las tres
rutas se evaluan sobre el mismo subconjunto de 275 imagenes.

In [2]:
import pandas as pd

from match.gallery import cargar_galeria_y_consultas

features_geometricas = pd.read_csv("data/features_mimic.csv")
ids_con_geometria = set(features_geometricas["sample_id"])

galeria, consultas = cargar_galeria_y_consultas("data/metadata.csv")
galeria = galeria[galeria["sample_id"].isin(ids_con_geometria)].reset_index(drop=True)
consultas = consultas[consultas["sample_id"].isin(ids_con_geometria)].reset_index(drop=True)

print(f"Galeria: {len(galeria)} imagenes | Consultas de evaluacion: {len(consultas)} imagenes")
galeria["clase"].value_counts()

Galeria: 236 imagenes | Consultas de evaluacion: 39 imagenes


clase
pulgar_arriba      88
saludo             82
mano_en_menton     10
neutral            10
brazos_cruzados     9
senalamiento        8
zen                 8
pensando            8
brazos_abiertos     8
manos_juntas        5
Name: count, dtype: int64

## 2. Ruta A: descriptores clasicos (HOG + LBP)

HOG captura la forma general (bordes y siluetas); LBP captura textura local. Se cachean
en CSV porque extraerlos de nuevo en cada corrida del notebook seria repetir trabajo.

In [3]:
import cv2
import numpy as np

from match.descriptors import extraer_hog, extraer_lbp_histograma

RUTA_DESCRIPTORES = Path("data/match_descriptores_clasicos.npz")

if RUTA_DESCRIPTORES.exists():
    datos = np.load(RUTA_DESCRIPTORES, allow_pickle=True)
    hog_galeria, lbp_galeria = datos["hog_galeria"], datos["lbp_galeria"]
    hog_consultas, lbp_consultas = datos["hog_consultas"], datos["lbp_consultas"]
else:
    def _extraer_lote(df):
        vectores_hog, vectores_lbp = [], []
        for ruta in df["ruta"]:
            imagen = cv2.imread(ruta)
            vectores_hog.append(extraer_hog(imagen))
            vectores_lbp.append(extraer_lbp_histograma(imagen))
        return np.array(vectores_hog), np.array(vectores_lbp)

    hog_galeria, lbp_galeria = _extraer_lote(galeria)
    hog_consultas, lbp_consultas = _extraer_lote(consultas)
    np.savez(
        RUTA_DESCRIPTORES,
        hog_galeria=hog_galeria, lbp_galeria=lbp_galeria,
        hog_consultas=hog_consultas, lbp_consultas=lbp_consultas,
    )

# Vector de la ruta A (ablation 1 compara esto contra usar solo HOG)
vectores_a_galeria = np.hstack([hog_galeria, lbp_galeria])
vectores_a_consultas = np.hstack([hog_consultas, lbp_consultas])
print("Ruta A lista:", vectores_a_galeria.shape, vectores_a_consultas.shape)

Ruta A lista: (236, 1790) (39, 1790)


## 3. Ruta B: geometria heredada de MIMIC

No se recalcula nada: se leen los mismos 10 valores geometricos por imagen que ya
produjo el notebook D1, indexados por `sample_id`.

In [4]:
from mimic.features import NOMBRES_FEATURES

geometria_por_id = features_geometricas.set_index("sample_id")[NOMBRES_FEATURES]

vectores_b_galeria = geometria_por_id.loc[galeria["sample_id"]].to_numpy()
vectores_b_consultas = geometria_por_id.loc[consultas["sample_id"]].to_numpy()
print("Ruta B lista:", vectores_b_galeria.shape, vectores_b_consultas.shape)

Ruta B lista: (236, 10) (39, 10)


## 4. Ruta C: embeddings SigLIP2

Representacion semantica moderna. Cargar el modelo tarda una vez (~10-40s); extraer el
embedding de cada imagen es rapido despues de eso. Se cachea en disco porque repetir la
extraccion en cada corrida seria el paso mas lento del notebook.

In [5]:
from match.embeddings import ExtractorSigLIP2

RUTA_EMBEDDINGS = Path("data/match_embeddings_siglip2.npz")

# El extractor se crea siempre (no solo cuando falta la cache): la
# seccion 7.2 (ablation de crop de persona) lo vuelve a necesitar mas
# adelante aunque los embeddings principales ya esten cacheados.
extractor_siglip2 = ExtractorSigLIP2()

if RUTA_EMBEDDINGS.exists():
    datos = np.load(RUTA_EMBEDDINGS)
    vectores_c_galeria = datos["galeria"]
    vectores_c_consultas = datos["consultas"]
else:
    def _extraer_embeddings(df):
        return np.array([
            extractor_siglip2.extraer(cv2.imread(ruta)) for ruta in df["ruta"]
        ])

    vectores_c_galeria = _extraer_embeddings(galeria)
    vectores_c_consultas = _extraer_embeddings(consultas)
    np.savez(RUTA_EMBEDDINGS, galeria=vectores_c_galeria, consultas=vectores_c_consultas)

print("Ruta C lista:", vectores_c_galeria.shape, vectores_c_consultas.shape)

C:\Users\migue\Downloads\computer_vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.


[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 7018.46it/s]

Ruta C lista: (236, 768) (39, 768)


## 5. Indexar y evaluar las tres rutas

Para cada ruta se construye un indice sobre la galeria y se buscan las 40 consultas de
test, reportando Recall@1, Recall@5 y MRR. El criterio de relevancia: un resultado
cuenta como correcto si su clase coincide con la clase real de la consulta.

In [6]:
import time

from match.evaluate import mrr, recall_at_k
from match.index import IndiceSimilitud

metadatos_galeria = [
    {"image_id": fila.sample_id, "etiqueta_pose": fila.clase, "ruta_archivo": fila.ruta}
    for fila in galeria.itertuples()
]
clases_verdaderas_consultas = consultas["clase"].tolist()


def evaluar_ruta(nombre, vectores_galeria, vectores_consultas):
    t0 = time.perf_counter()
    indice = IndiceSimilitud(vectores_galeria, metadatos_galeria)
    latencia_indexacion = time.perf_counter() - t0

    t0 = time.perf_counter()
    resultados_por_consulta = [indice.buscar(v, k=5) for v in vectores_consultas]
    latencia_consulta_media = (time.perf_counter() - t0) / len(vectores_consultas)

    return {
        "ruta": nombre,
        "Recall@1": recall_at_k(resultados_por_consulta, clases_verdaderas_consultas, k=1),
        "Recall@5": recall_at_k(resultados_por_consulta, clases_verdaderas_consultas, k=5),
        "MRR": mrr(resultados_por_consulta, clases_verdaderas_consultas),
        "latencia_indexacion_ms": latencia_indexacion * 1000,
        "latencia_consulta_ms": latencia_consulta_media * 1000,
    }, resultados_por_consulta

resultado_a, resultados_por_consulta_a = evaluar_ruta("A: HOG+LBP", vectores_a_galeria, vectores_a_consultas)
resultado_b, resultados_por_consulta_b = evaluar_ruta("B: geometria", vectores_b_galeria, vectores_b_consultas)
resultado_c, resultados_por_consulta_c = evaluar_ruta("C: SigLIP2", vectores_c_galeria, vectores_c_consultas)

tabla_comparativa = pd.DataFrame([resultado_a, resultado_b, resultado_c]).round(3)
tabla_comparativa

,ruta,Recall@1,Recall@5,MRR,latencia_indexacion_ms,latencia_consulta_ms
0,A: HOG+LBP,0.538,0.744,0.623,7.325,0.549
1,B: geometria,0.436,0.795,0.569,0.165,0.151
2,C: SigLIP2,0.846,1.000,0.907,8.531,0.325


## 6. Consultas faciles, dificiles y ambiguas

El promedio de Recall esconde historias distintas. Se separan las consultas en tres
grupos para explicar los resultados en vez de solo reportarlos:

- **Faciles**: clases de HaGRID (`saludo`, `pulgar_arriba`), con mucho soporte en galeria.
- **Dificiles**: clases curadas chicas (7-16 imagenes en galeria).
- **Ambiguas**: pares que la matriz de confusion de MIMIC ya mostro que se confunden.

In [7]:
CLASES_FACILES = {"saludo", "pulgar_arriba"}
CLASES_AMBIGUAS = {"pensando", "mano_en_menton", "zen", "manos_juntas"}


def categoria_de(clase):
    if clase in CLASES_FACILES:
        return "facil"
    if clase in CLASES_AMBIGUAS:
        return "ambigua"
    return "dificil"

consultas_categorizadas = consultas.copy()
consultas_categorizadas["categoria"] = consultas_categorizadas["clase"].map(categoria_de)

filas_por_categoria = []
for nombre_ruta, resultados_por_consulta in [
    ("A: HOG+LBP", resultados_por_consulta_a),
    ("B: geometria", resultados_por_consulta_b),
    ("C: SigLIP2", resultados_por_consulta_c),
]:
    for categoria in ["facil", "dificil", "ambigua"]:
        indices = consultas_categorizadas.index[consultas_categorizadas["categoria"] == categoria].tolist()
        if not indices:
            continue
        subconjunto_resultados = [resultados_por_consulta[i] for i in indices]
        subconjunto_clases = [clases_verdaderas_consultas[i] for i in indices]
        filas_por_categoria.append({
            "ruta": nombre_ruta,
            "categoria": categoria,
            "n_consultas": len(indices),
            "Recall@1": recall_at_k(subconjunto_resultados, subconjunto_clases, k=1),
        })

pd.DataFrame(filas_por_categoria).pivot(index="ruta", columns="categoria", values="Recall@1").round(2)

categoria,ambigua,dificil,facil
ruta,,,
A: HOG+LBP,0.00,0.25,0.71
B: geometria,0.29,0.00,0.54
C: SigLIP2,0.57,0.50,0.96


## 7. Ablation study

Se retira o modifica un componente a la vez para medir su efecto sobre Recall@1.

### 7.1 Textura: HOG solo vs HOG + LBP

In [8]:
resultado_hog_solo, _ = evaluar_ruta("A sin textura (solo HOG)", hog_galeria, hog_consultas)
resultado_hog_lbp, _ = evaluar_ruta("A con textura (HOG+LBP)", vectores_a_galeria, vectores_a_consultas)

pd.DataFrame([resultado_hog_solo, resultado_hog_lbp])[["ruta", "Recall@1", "Recall@5", "MRR"]]

,ruta,Recall@1,Recall@5,MRR
0,A sin textura (solo HOG),0.538462,0.74359,0.62265
1,A con textura (HOG+LBP),0.538462,0.74359,0.62265


### 7.2 Crop de persona vs frame completo (ruta C)

Se recorta cada imagen alrededor de la persona (usando el `bbox_persona` que ahora
calcula MIMIC) antes de pasarla por SigLIP2, y se compara contra usar el frame completo.
Requiere volver a correr el detector de landmarks, esta vez solo para obtener el
bounding box -- no para las features geometricas, que ya estan cacheadas.

In [9]:
from mimic.features import calcular_bbox_persona
from mimic.landmarks import DetectorHolistic

RUTA_EMBEDDINGS_CROP = Path("data/match_embeddings_siglip2_crop.npz")


def _recortar_persona(imagen, detector):
    resultado_landmarks = detector.procesar(imagen)
    if resultado_landmarks.pose is None:
        return imagen  # si no se detecta a nadie, se usa el frame completo
    alto, ancho = imagen.shape[:2]
    x, y, w, h = calcular_bbox_persona(resultado_landmarks.pose, ancho, alto)
    if w <= 0 or h <= 0:
        return imagen
    return imagen[y:y + h, x:x + w]

if RUTA_EMBEDDINGS_CROP.exists():
    datos = np.load(RUTA_EMBEDDINGS_CROP)
    vectores_c_crop_galeria = datos["galeria"]
    vectores_c_crop_consultas = datos["consultas"]
else:
    detector_crop = DetectorHolistic()

    def _extraer_embeddings_recortados(df):
        vectores = []
        for ruta in df["ruta"]:
            imagen = cv2.imread(ruta)
            recorte = _recortar_persona(imagen, detector_crop)
            vectores.append(extractor_siglip2.extraer(recorte))
        return np.array(vectores)

    vectores_c_crop_galeria = _extraer_embeddings_recortados(galeria)
    vectores_c_crop_consultas = _extraer_embeddings_recortados(consultas)
    np.savez(RUTA_EMBEDDINGS_CROP, galeria=vectores_c_crop_galeria, consultas=vectores_c_crop_consultas)

resultado_c_completo, _ = evaluar_ruta("C frame completo", vectores_c_galeria, vectores_c_consultas)
resultado_c_crop, _ = evaluar_ruta("C crop de persona", vectores_c_crop_galeria, vectores_c_crop_consultas)

pd.DataFrame([resultado_c_completo, resultado_c_crop])[["ruta", "Recall@1", "Recall@5", "MRR"]]

,ruta,Recall@1,Recall@5,MRR
0,C frame completo,0.846154,1.000000,0.906838
1,C crop de persona,0.794872,0.923077,0.840598


### 7.3 Rutas individuales (resumen)

Ya calculado en la seccion 5 -- se repite aqui como parte del ablation por ser el punto
de comparacion base de los otros tres experimentos.

In [10]:
tabla_comparativa[["ruta", "Recall@1", "Recall@5", "MRR"]]

,ruta,Recall@1,Recall@5,MRR
0,A: HOG+LBP,0.538,0.744,0.623
1,B: geometria,0.436,0.795,0.569
2,C: SigLIP2,0.846,1.000,0.907


### 7.4 Re-ranking: la mejor ruta + geometria (experimento E-D)

Se toma el Top-10 de la ruta con mejor Recall@1 y se reordena por similitud geometrica,
para ver si combinar semantica visual con geometria de pose mejora el Recall@1 final.

In [11]:
from match.reranking import reordenar_por_geometria

mejor_ruta = tabla_comparativa.sort_values("Recall@1", ascending=False).iloc[0]["ruta"]
print("Mejor ruta individual:", mejor_ruta)

vectores_geometricos_por_id = {
    fila.sample_id: geometria_por_id.loc[fila.sample_id].to_numpy()
    for fila in galeria.itertuples()
}

vectores_mejor_ruta_galeria = {
    "A: HOG+LBP": vectores_a_galeria, "B: geometria": vectores_b_galeria, "C: SigLIP2": vectores_c_galeria,
}[mejor_ruta]
vectores_mejor_ruta_consultas = {
    "A: HOG+LBP": vectores_a_consultas, "B: geometria": vectores_b_consultas, "C: SigLIP2": vectores_c_consultas,
}[mejor_ruta]

indice_mejor_ruta = IndiceSimilitud(vectores_mejor_ruta_galeria, metadatos_galeria)
resultados_top10 = [indice_mejor_ruta.buscar(v, k=10) for v in vectores_mejor_ruta_consultas]
resultados_reordenados = [
    reordenar_por_geometria(top10, vector_consulta_geo, vectores_geometricos_por_id)
    for top10, vector_consulta_geo in zip(resultados_top10, vectores_b_consultas)
]

recall1_sin_reranking = recall_at_k(resultados_top10, clases_verdaderas_consultas, k=1)
recall1_con_reranking = recall_at_k(resultados_reordenados, clases_verdaderas_consultas, k=1)

pd.DataFrame([
    {"variante": f"{mejor_ruta} sin re-ranking", "Recall@1": recall1_sin_reranking},
    {"variante": f"{mejor_ruta} + re-ranking geometrico", "Recall@1": recall1_con_reranking},
])

Mejor ruta individual: C: SigLIP2


,variante,Recall@1
0,C: SigLIP2 sin re-ranking,0.846154
1,C: SigLIP2 + re-ranking geometrico,0.769231


## 8. Demo Gradio

La pestana MATCH de la misma app de MIMIC: sube o captura una foto y devuelve el Top-5
mas parecido de la galeria.

In [12]:
from app.app import construir_demo

construir_demo().launch(share=EN_COLAB, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860


* To create a public link, set `share=True` in `launch()`.


## Continuidad hacia la Clase 3 (ASK)

Se conservan los embeddings, el indice y la interfaz. La Clase 3 anadira una capa de
eventos temporales y grounding por lenguaje natural (LocateAnything-3B) sobre video,
sin modificar MIMIC ni MATCH.